# Submissão A — NumPy DNN-2

Classificação do `subm1.csv` com o nosso melhor modelo implementado em NumPy.

In [1]:
import re, copy, pickle
from abc import ABCMeta, abstractmethod
import numpy as np
import pandas as pd


In [ ]:
class TFIDFVectorizerNumpy:
    def __init__(self, max_features=5000, min_df=5, ngram_range=(1,1)):
        self.max_features = max_features
        self.min_df       = min_df
        self.ngram_range  = ngram_range
        self.vocab_       = {}
        self.idf_         = None
        self._tok_re      = re.compile(r"[a-z]{2,}")

    def _tokenize(self, text):
        tokens = self._tok_re.findall(text.lower())
        grams, lo, hi = [], *self.ngram_range
        for n in range(lo, hi + 1):
            for i in range(len(tokens) - n + 1):
                grams.append(' '.join(tokens[i:i+n]))
        return grams

    def transform(self, corpus):
        import scipy.sparse as sp
        from sklearn.preprocessing import normalize
        V = len(self.vocab_)
        X = sp.lil_matrix((len(corpus), V), dtype=np.float32)
        for row, doc in enumerate(corpus):
            tok_freq = {}
            for tok in self._tokenize(doc):
                if tok in self.vocab_:
                    tok_freq[tok] = tok_freq.get(tok, 0) + 1
            for tok, freq in tok_freq.items():
                X[row, self.vocab_[tok]] = np.log1p(freq)
        X = X.tocsr().multiply(self.idf_)
        return normalize(X, norm='l2', axis=1)

print('TFIDFVectorizerNumpy definida.')


TFIDFVectorizerNumpy definida.


In [3]:
class Layer(metaclass=ABCMeta):
    @abstractmethod
    def forward_propagation(self, inputs, training): raise NotImplementedError
    def set_input_shape(self, s): self._input_shape = s
    def input_shape(self): return self._input_shape
    def output_shape(self): raise NotImplementedError
    def parameters(self): return 0

class DenseLayer(Layer):
    def __init__(self, n_units, input_shape=None, l2=0.0, l1=0.0):
        self.n_units = n_units
        self._input_shape = input_shape
        self.weights = None
        self.biases  = None
    def initialize(self, optimizer):
        fan_in = self.input_shape()[0]
        self.weights = np.random.randn(fan_in, self.n_units).astype(np.float32) * np.sqrt(2.0/fan_in)
        self.biases  = np.zeros((1, self.n_units), dtype=np.float32)
        return self
    def forward_propagation(self, inputs, training):
        return inputs.dot(self.weights) + self.biases
    def output_shape(self): return (self.n_units,)
    def parameters(self): return int(np.prod(self.weights.shape) + np.prod(self.biases.shape))

class DropoutLayer(Layer):
    def __init__(self, rate=0.5):
        self.rate = rate
    def initialize(self, optimizer): return self
    def forward_propagation(self, inputs, training):
        return inputs  # sem dropout na inferência
    def output_shape(self): return self._input_shape
    def parameters(self): return 0

print('DenseLayer, DropoutLayer definidas.')


DenseLayer, DropoutLayer definidas.


In [4]:
class ReLUActivation(Layer):
    def initialize(self, optimizer): return self
    def forward_propagation(self, inputs, training):
        return np.maximum(0, inputs)
    def output_shape(self): return self._input_shape
    def parameters(self): return 0

class SoftmaxActivation(Layer):
    def initialize(self, optimizer): return self
    def forward_propagation(self, inputs, training):
        e = np.exp(inputs - inputs.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)
    def output_shape(self): return self._input_shape
    def parameters(self): return 0

print('ReLUActivation, SoftmaxActivation definidas.')


ReLUActivation, SoftmaxActivation definidas.


In [ ]:
class Adam:
    def __init__(self, learning_rate=1e-3): pass

class NeuralNetwork:
    """Versão mínima para inferência, sem treino."""
    def __init__(self, optimizer=None, loss=None, verbose=False, **kwargs):
        self.optimizer  = optimizer or Adam()
        self.layers     = []
        self.batch_size = 1024

    def add(self, layer):
        if self.layers:
            layer.set_input_shape(self.layers[-1].output_shape())
        if hasattr(layer, 'initialize'):
            layer.initialize(copy.deepcopy(self.optimizer))
        self.layers.append(layer)

    def fit(self, X, y, *args, **kwargs):
        """Chamado apenas para forçar inicialização dos pesos, não treina."""
        pass

    def predict(self, X):
        out = X.toarray() if hasattr(X, 'toarray') else X
        for layer in self.layers:
            out = layer.forward_propagation(out, training=False)
        return out

    def predict_classes(self, X):
        return np.argmax(self.predict(X), axis=1)

print('NeuralNetwork (inferência) definida.')


NeuralNetwork (inferência) definida.


In [6]:
def load_nn_weights(model, filename):
    data = np.load(f'{filename}.npz', allow_pickle=True)
    idx  = 0
    for layer in model.layers:
        if hasattr(layer, 'weights') and layer.weights is not None:
            layer.weights = data[f'param_{idx}']
            layer.biases  = data[f'param_{idx+1}']
            idx += 2
    print(f'Pesos carregados de {filename}.npz  ({idx} arrays)')


## Classificação e geração do ficheiro de submissão

In [ ]:
# -- 1. Carregar artefactos ----------------------------------------------------
with open('numpy_tfidf.pkl', 'rb') as f:
    tfidf_loaded = pickle.load(f)

CLASSES      = ['Human', 'Google', 'Anthropic', 'Meta', 'OpenAI']
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASSES)}
N_CLASSES    = len(CLASSES)

# -- 2. Carregar subm1.csv -----------------------------------------------------
df = pd.read_csv('subm1.csv', sep=';', encoding='utf-8-sig')
df.columns = df.columns.str.strip()
print(f'Registos: {len(df)}')

# -- 3. Vectorizar -------------------------------------------------------------
X = tfidf_loaded.transform(df['Text'].tolist())
print(f'Features: {X.shape}')

# -- 4. Reconstruir rede e carregar pesos --------------------------------------
INPUT_DIM = X.shape[1]

model = NeuralNetwork(optimizer=Adam(), verbose=False)
model.add(DenseLayer(256, input_shape=(INPUT_DIM,)))
model.add(ReLUActivation())
model.add(DropoutLayer(0.5))
model.add(DenseLayer(128))
model.add(ReLUActivation())
model.add(DropoutLayer(0.3))
model.add(DenseLayer(N_CLASSES))
model.add(SoftmaxActivation())

load_nn_weights(model, 'numpy_dnn2')
print('Modelo carregado.')

# -- 5. Previsões ---------------------------------------------------------------
y_pred      = model.predict_classes(X)
labels_pred = [IDX_TO_CLASS[i] for i in y_pred]

# -- 6. Guardar CSV  ------------------------------------------------------------
df_out = pd.DataFrame({'ID': df['ID'], 'Labels': labels_pred})
df_out.to_csv('subm1-g5-MECD-A.csv', index=False, sep=';')
print('\nFicheiro guardado: subm1-g5-MECD-A.csv')
print(df_out['Labels'].value_counts().to_string())
print(df_out.head(10).to_string(index=False))


Registos: 150
Features: (150, 20000)
Pesos carregados de numpy_dnn2.npz  (6 arrays)
Modelo carregado.

Ficheiro guardado: subm1-g5-MECD-A.csv
Labels
OpenAI       43
Anthropic    36
Google       31
Human        25
Meta         15
   ID    Labels
 D2-1 Anthropic
 D2-2 Anthropic
 D2-3    Google
 D2-4      Meta
 D2-5     Human
 D2-6 Anthropic
 D2-7    Google
 D2-8    Google
 D2-9    Google
D2-10    OpenAI
